In [11]:
import yaml
from pathlib import Path
from sim.generate_basket import generate_basket, trx_by_day_of_week

In [14]:
def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

sim = load_yaml("./config/simulation.yaml")["simulation"]
sim

{'transaction_volume': {'weekday_base': {'monday': 40,
   'tuesday': 35,
   'wednesday': 45,
   'thursday': 35,
   'friday': 65,
   'saturday': 90,
   'sunday': 85},
  'noise': {'distribution': 'poisson', 'multiplier': 1.0}},
 'basket': {'min_items': 1, 'max_items': 8},
 'quantity_model': {'base_lambda': 1.2},
 'price_variation': {'std_pct': 0.08}}

In [19]:
def trx_by_day_of_week(day_of_week):
    return sim['transaction_volume']['weekday_base'][day_of_week]

In [20]:
trx_by_day_of_week('monday')

40

In [9]:
from sim.generate_date import date_list, day_of_week
from datetime import datetime

In [12]:
date = date_list("2026-01-12", "2026-01-13")[0]
datetime.strptime(date, "%Y-%m-%d").strftime("%B").lower()

'january'

In [2]:
import pandas as pd
import yaml

In [12]:
with open("./config/funnel.yaml", "r") as f:
    funnel_config = yaml.safe_load(f)

In [13]:
funnel_config

{'default_tiers': {'funnel_steps': {'landing_page': {'conversion_rate': 0.3,
    'duration': [8, 35]},
   'product_view': {'conversion_rate': 0.3, 'duration': [33, 615]},
   'add_to_cart': {'conversion_rate': 0.2, 'duration': [56, 239]},
   'checkout': {'conversion_rate': 0.7, 'duration': [47, 264]},
   'paid': {'conversion_rate': 1, 'duration': [0, 0]}}},
 'tiers': [{'funnel_steps': {'landing_page': {'conversion_rate': 0.3,
     'duration': [8, 35]},
    'product_view': {'conversion_rate': 0.3, 'duration': [33, 615]},
    'add_to_cart': {'conversion_rate': 0.2, 'duration': [56, 239]},
    'checkout': {'conversion_rate': 0.7, 'duration': [47, 264]},
    'paid': {'conversion_rate': 1, 'duration': [0, 0]}},
   'id': 1,
   'label': 'High spenders & bulk buyers'}]}

In [ ]:
from pathlib import Path
import yaml

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def deep_merge(base, override):
    result = dict(base)
    for k, v in override.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            result[k] = deep_merge(base[k], v)
        else:
            result[k] = v
    return result

def load_all_categories(base_defaults):
    categories = {}
    for file in Path("config/categories").glob("*.yaml"):
        data = load_yaml(file)
        categories[data["category"]] = deep_merge(
            base_defaults, data
        )
    return categories


In [5]:
df = pd.read_excel('./config/product-config/catalog.xlsx')
df

,category,subcategory,product
0,vegetables,leafy_greens,"spinach, kale, lettuce"
1,vegetables,root_vegetables,"carrots, potatoes, beets"
2,fruits,citrus,"oranges, lemons, limes"
3,fruits,berries,"strawberries, blueberries, raspberries"
4,fruits,tropical,"mangoes, pineapples, bananas"
5,fruits,stone_fruits,"peaches, plums, cherries"
6,fruits,melons,"watermelon, cantaloupe, honeydew"
7,grains,rice,"white_rice_5kg, brown_rice_1kg, premium_white_..."
8,dairy,milk,"whole_milk_1L, skim_milk_1L, almond_milk_1L"
9,dairy,cheese,"cheddar_cheese_200g, mozzarella_cheese_200g"


In [14]:
expanded_df = (
    df
    .assign(product=df['product'].str.split(', ', regex=True))
    .explode('product')
    .reset_index(drop=True)
)

expanded_df['category'] = expanded_df['category'].str.strip().str.lower()
expanded_df['subcategory'] = expanded_df['subcategory'].str.strip().str.lower()
expanded_df['product'] = expanded_df['product'].str.strip().str.lower()

expanded_df


,category,subcategory,product
0,vegetables,leafy_greens,spinach
1,vegetables,leafy_greens,kale
2,vegetables,leafy_greens,lettuce
3,vegetables,root_vegetables,carrots
4,vegetables,root_vegetables,potatoes
...,...,...,...
119,condiments_seasonings,dressings,italian_dressing_250ml
120,condiments_seasonings,dressings,vinaigrette_250ml
121,condiments_seasonings,marinades,teriyaki_marinade_250ml
122,condiments_seasonings,marinades,barbecue_marinade_250ml
